# mPES — Entrenamiento en Colab Pro+

Entrena **pes_dqn** (DQN) o **pes_ac** (A2C) en una instancia GPU de Colab Pro+.
Los modelos entrenados se copian automáticamente a Google Drive.

> Los modelos son pequeños (32–64 neuronas). La GPU no acelera mucho,
> pero la instancia GPU trae más RAM y mejor CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title **Configuración** { display-mode: "form" }

PACKAGE = "pes_ac" #@param ["pes_dqn", "pes_ac"]
NUM_EPISODES = 50000 #@param {type:"integer"}
BRANCH = "dqn_and_ac" #@param {type:"string"}
GITHUB_TOKEN = "Token_2026" #@param {type:"string"}
NTFY_TOPIC = "mpes_opt" #@param {type:"string"}

# --- Derivados ---
import os
from datetime import datetime

GITHUB_REPO = 'https://github.com/Maximiliano0/mPES_2026.git'
_MODULE_MAP = {'pes_dqn': 'train_dqn', 'pes_ac': 'train_ac'}
_SUFFIX_MAP = {'pes_dqn': 'DQN_TRAIN', 'pes_ac': 'AC_TRAIN'}
TRAIN_MODULE = _MODULE_MAP[PACKAGE]
REPO_DIR = '/content/mPES'
DRIVE_BACKUP = f'/content/drive/MyDrive/mPES_backups/{PACKAGE}'
TRAIN_DATE = datetime.now().strftime('%Y-%m-%d')
TRAIN_SUBDIR = f'{PACKAGE}/inputs/{TRAIN_DATE}_{_SUFFIX_MAP[PACKAGE]}'

os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f'Paquete: {PACKAGE}  |  Modulo: {PACKAGE}.ext.{TRAIN_MODULE}')
print(f'Episodios: {NUM_EPISODES:,}  |  Rama: {BRANCH}')
print(f'Backup: {DRIVE_BACKUP}')

## 1. Setup del entorno

In [ ]:
import os, subprocess

# --- Clonar o actualizar repo ---
if not os.path.exists(REPO_DIR):
    repo_url = GITHUB_REPO
    if GITHUB_TOKEN:
        repo_url = GITHUB_REPO.replace('https://', f'https://{GITHUB_TOKEN}@')
    subprocess.run(['git', 'clone', '--branch', BRANCH, repo_url, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

# --- Dependencias ---
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.0',
    'optuna==4.7.0', 'gym==0.26.1', 'pygame==2.5.2', 'rich==14.3.2',
    'mne>=1.6.1', 'scipy>=1.14.0', 'colorlog', 'colorama'
], check=True)

# --- Verificar numpy 2.x (requerido por TF de Colab) ---
import numpy
_nv = numpy.__version__
if int(_nv.split('.')[0]) < 2:
    raise RuntimeError(
        f'numpy {_nv} es 1.x -- reinicia el runtime (Runtime > Restart session) '
        f'y volve a ejecutar todas las celdas.'
    )

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'numpy {_nv}, TF {tf.__version__}')
print(f'GPU: {gpus[0].name if gpus else "ninguna (CPU)"}')

In [ ]:
import subprocess, os

subprocess.run(['nvidia-smi'], check=False)

with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line:
            print(f'\nRAM total: {int(line.split()[1]) / 1024**2:.1f} GB')
            break

# --- Verificar archivos requeridos ---
csv1 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'sequence_lengths.csv')
csv2 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'initial_severity.csv')

print('\nArchivos requeridos:')
for fpath in [csv1, csv2]:
    ok = os.path.exists(fpath)
    print(f'  {"OK" if ok else "FALTA"}  {os.path.relpath(fpath, REPO_DIR)}')

## 2. Entrenar

Lanza el entrenamiento como subproceso con GPU habilitada.
El script genera el modelo `.keras`, rewards `.npy`, plots y config.

In [ ]:
import subprocess, os, time
import urllib.request


def ntfy(msg, title=None, priority='default', tags=''):
    """Envia notificacion push via ntfy.sh."""
    if not NTFY_TOPIC:
        return
    try:
        headers = {}
        if title:
            safe = title.encode('latin-1', errors='ignore').decode('latin-1').strip()
            if safe:
                headers['Title'] = safe
        if priority and priority != 'default':
            headers['Priority'] = priority
        if tags:
            headers['Tags'] = tags
        req = urllib.request.Request(
            f'https://ntfy.sh/{NTFY_TOPIC}',
            data=msg.encode('utf-8'), headers=headers, method='POST'
        )
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print(f'  [ntfy] Error: {e}', flush=True)


def fmt_dur(s):
    """Segundos a formato legible."""
    if s < 60: return f'{s:.0f}s'
    if s < 3600: return f'{int(s//60)}m {int(s%60)}s'
    return f'{int(s//3600)}h {int((s%3600)//60)}m'


# --- Entorno y comando ---
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'VIRTUAL_ENV': '/content',
    'PYTHONUNBUFFERED': '1',
    'TF_ENABLE_ONEDNN_OPTS': '0',
    'TF_CPP_MIN_LOG_LEVEL': '2',
})

cmd = ['python3', '-m', f'{PACKAGE}.ext.{TRAIN_MODULE}', str(NUM_EPISODES)]

ntfy(
    f'Entrenamiento iniciado\nPaquete: {PACKAGE}\nEpisodios: {NUM_EPISODES:,}',
    title=f'{PACKAGE} -- Train iniciado', tags='rocket'
)

print(f'> {" ".join(cmd)}')
print(f'  cwd: {REPO_DIR}  |  Episodios: {NUM_EPISODES:,}')
print()

# --- Ejecutar ---
t_start = time.time()

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

exit_code = proc.wait()
elapsed = time.time() - t_start

print(f'\n{"="*50}')
print(f'  FINALIZADO  |  Tiempo: {fmt_dur(elapsed)}  |  Exit: {exit_code}')
print(f'{"="*50}')

ntfy(
    f'Entrenamiento finalizado\nPaquete: {PACKAGE}\n'
    f'Episodios: {NUM_EPISODES:,}\nTiempo: {fmt_dur(elapsed)}\nExit: {exit_code}',
    title=f'{PACKAGE} -- Train finalizado',
    priority='high',
    tags='white_check_mark' if exit_code == 0 else 'warning'
)

## 3. Copiar resultados a Drive

In [ ]:
import os, shutil

train_dir_src = os.path.join(REPO_DIR, TRAIN_SUBDIR)
train_dir_dst = os.path.join(DRIVE_BACKUP, f'{TRAIN_DATE}_{_SUFFIX_MAP[PACKAGE]}')

if os.path.exists(train_dir_src):
    shutil.copytree(train_dir_src, train_dir_dst, dirs_exist_ok=True)
    print(f'Outputs copiados a: {train_dir_dst}')
    for fname in sorted(os.listdir(train_dir_dst)):
        size = os.path.getsize(os.path.join(train_dir_dst, fname))
        print(f'  {fname} ({size/1024:.0f} KB)')
else:
    print(f'No se encontro directorio de salida: {train_dir_src}')
    print('Directorios disponibles:')
    inputs_dir = os.path.join(REPO_DIR, PACKAGE, 'inputs')
    for d in sorted(os.listdir(inputs_dir)):
        if os.path.isdir(os.path.join(inputs_dir, d)):
            print(f'  {d}')